In [1]:
!pip install sentence-transformers

In [2]:
import sys
!{sys.executable} -m pip install faiss-cpu

In [3]:
# =========================
# CORE LIBRARIES
# =========================

import pandas as pd
import numpy as np

# =========================
# NLP + EMBEDDINGS
# =========================

from sentence_transformers import SentenceTransformer

# =========================
# MACHINE LEARNING
# =========================

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

# =========================
# VECTOR SEARCH
# =========================

import faiss

# =========================
# PYTORCH
# =========================

import torch
import torch.nn as nn
import torch.nn.functional as F

# =========================
# UTILITIES
# =========================

from collections import defaultdict
from datetime import datetime
import pickle
import warnings

warnings.filterwarnings("ignore")


Load the dating profile dataset.

In [4]:
# =========================
# LOAD DATASET
# =========================

df = pd.read_csv(
    "okcupid_profiles.csv"
)

print(df.shape)

df.head()

(59946, 31)


,age,status,sex,orientation,body_type,diet,drinks,drugs,education,ethnicity,...,essay0,essay1,essay2,essay3,essay4,essay5,essay6,essay7,essay8,essay9
0,22,single,m,straight,a little extra,strictly anything,socially,never,working on college/university,"asian, white",...,about me: i would love to think that i was so...,currently working as an international agent fo...,making people laugh. ranting about a good salt...,"the way i look. i am a six foot half asian, ha...","books: absurdistan, the republic, of mice and ...",food. water. cell phone. shelter.,duality and humorous things,trying to find someone to hang out with. i am ...,i am new to california and looking for someone...,you want to be swept off your feet! you are ti...
1,35,single,m,straight,average,mostly other,often,sometimes,working on space camp,white,...,i am a chef: this is what that means. 1. i am ...,dedicating everyday to being an unbelievable b...,being silly. having ridiculous amonts of fun w...,NaN,i am die hard christopher moore fan. i don't r...,delicious porkness in all of its glories. my b...,NaN,NaN,i am very open and will share just about anyth...,NaN
2,38,available,m,straight,thin,anything,socially,NaN,graduated from masters program,NaN,...,"i'm not ashamed of much, but writing public te...","i make nerdy software for musicians, artists, ...",improvising in different contexts. alternating...,my large jaw and large glasses are the physica...,okay this is where the cultural matrix gets so...,movement conversation creation contemplation t...,NaN,viewing. listening. dancing. talking. drinking...,"when i was five years old, i was known as ""the...","you are bright, open, intense, silly, ironic, ..."
3,23,single,m,straight,thin,vegetarian,socially,NaN,working on college/university,white,...,i work in a library and go to school. . .,reading things written by old dead people,playing synthesizers and organizing books acco...,socially awkward but i do my best,"bataille, celine, beckett. . . lynch, jarmusch...",NaN,cats and german philosophy,NaN,NaN,you feel so inclined.
4,29,single,m,straight,athletic,NaN,socially,never,graduated from college/university,"asian, black, other",...,hey how's it going? currently vague on the pro...,work work work work + play,creating imagery to look at: http://bagsbrown....,i smile a lot and my inquisitive nature,"music: bands, rappers, musicians at the moment...",NaN,NaN,NaN,NaN,NaN


In [5]:
# =========================
# IMPORTANT FEATURES
# =========================

selected_columns = [

    # demographics
    "age",
    "sex",
    "orientation",
    "status",
    "body_type",

    # lifestyle
    "diet",
    "drinks",
    "drugs",

    # education/work
    "education",
    "job",

    # location
    "location",

    # essays
    "essay0",
    "essay1",
    "essay2",
    "essay3",
    "essay4",
    "essay5",
    "essay6",
    "essay7",
    "essay8",
    "essay9"
]

df = df[
    selected_columns
]

print(df.shape)

(59946, 21)


# Step 4 — Handle Missing Values

Clean incomplete user profiles.

In [6]:
# =========================
# FILL TEXT FEATURES
# =========================

essay_columns = [

    "essay0",
    "essay1",
    "essay2",
    "essay3",
    "essay4",
    "essay5",
    "essay6",
    "essay7",
    "essay8",
    "essay9"
]

for col in essay_columns:

    df[col] = df[col].fillna(
        ""
    )

# =========================
# FILL CATEGORICAL FEATURES
# =========================

categorical_columns = [

    "sex",
    "orientation",
    "status",
    "body_type",
    "diet",
    "drinks",
    "drugs",
    "education",
    "job",
    "location"
]

for col in categorical_columns:

    df[col] = df[col].fillna(
        "unknown"
    )

# =========================
# FILL NUMERIC FEATURES
# =========================

df["age"] = df["age"].fillna(

    df["age"].median()
)

# Step 5 — Combine Essay Fields

Merge all essay sections into a single semantic profile.

In [7]:
# =========================
# COMBINE ESSAYS
# =========================

df["combined_essays"] = (

    df[
        essay_columns
    ]

    .astype(str)

    .agg(

        " ".join,

        axis=1
    )
)

df[
    "combined_essays"
].head()

0    about me:  i would love to think that i was so...
1    i am a chef: this is what that means. 1. i am ...
2    i'm not ashamed of much, but writing public te...
3    i work in a library and go to school. . . read...
4    hey how's it going? currently vague on the pro...
Name: combined_essays, dtype: object

# Step 6 — Clean Essay Text

Normalize essay text before embedding generation.

In [8]:
# =========================
# TEXT CLEANING
# =========================

import re

def clean_text(text):

    text = text.lower()

    text = re.sub(

        r"[^a-zA-Z\s]",

        " ",

        text
    )

    text = re.sub(

        r"\s+",

        " ",

        text
    )

    return text.strip()

df["combined_essays"] = (

    df[
        "combined_essays"
    ]

    .apply(clean_text)
)

df["combined_essays"].head()

0    about me i would love to think that i was some...
1    i am a chef this is what that means i am a wor...
2    i m not ashamed of much but writing public tex...
3    i work in a library and go to school reading t...
4    hey how s it going currently vague on the prof...
Name: combined_essays, dtype: object

# Step 7 — Encode Categorical Features

Convert categorical profile data into numerical form.

In [9]:
# =========================
# LABEL ENCODING
# =========================

label_encoders = {}

for col in categorical_columns:

    encoder = LabelEncoder()

    df[col] = encoder.fit_transform(
        df[col]
    )

    label_encoders[col] = encoder

# Step 8 — Normalize Numeric Features

Scale numerical profile attributes.

In [10]:
# =========================
# SCALE AGE
# =========================

scaler = StandardScaler()

df["age"] = scaler.fit_transform(

    df[["age"]]
)

# Step 9 — Save Processed Dataset

Persist cleaned profiles for future pipeline stages.

In [11]:
# =========================
# SAVE CLEAN DATA
# =========================

df.to_csv(

    "processed_okcupid.csv",

    index=False
)

print(
    "Processed dataset saved."
)

Processed dataset saved.


# Step 10 — Load Embedding Model

Load a production-grade semantic embedding model.

In [12]:
# =========================
# LOAD EMBEDDING MODEL
# =========================

embedding_model = SentenceTransformer(

    "all-MiniLM-L6-v2"
)

print(
    "Embedding model loaded."
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


# Step 11 — Generate Semantic Essay Embeddings

Convert user essays into semantic personality vectors.

In [13]:
# =========================
# GENERATE ESSAY EMBEDDINGS
# =========================

essay_embeddings = embedding_model.encode(

    df["combined_essays"].tolist(),

    batch_size=64,

    show_progress_bar=True,

    convert_to_numpy=True
)

print(
    essay_embeddings.shape
)

Batches:   0%|          | 0/937 [00:00<?, ?it/s]

(59946, 384)


# Step 12 — Save Semantic Embeddings

Persist semantic vectors for future retrieval/ranking stages.

In [14]:
# =========================
# SAVE EMBEDDINGS
# =========================

np.save(

    "essay_embeddings.npy",

    essay_embeddings
)

print(
    "Essay embeddings saved."
)

Essay embeddings saved.


# Step 13 — Build Structured Profile Features

Prepare structured dating attributes for recommendation learning.

In [15]:
# =========================
# STRUCTURED FEATURES
# =========================

structured_features = [

    "age",
    "sex",
    "orientation",
    "status",
    "body_type",
    "diet",
    "drinks",
    "drugs",
    "education",
    "job",
    "location"
]

profile_matrix = df[
    structured_features
].values.astype("float32")

print(
    profile_matrix.shape
)

(59946, 11)


# Step 14 — Neural Profile Encoder

Learn dense profile compatibility embeddings.

In [16]:
# =========================
# PROFILE ENCODER
# =========================

class ProfileEncoder(

    nn.Module
):

    def __init__(

        self,

        input_dim,

        embedding_dim=128
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(
                256,
                embedding_dim
            )
        )

    def forward(

        self,

        x
    ):

        x = self.network(x)

        return F.normalize(

            x,

            p=2,

            dim=-1
        )

# Step 15 — Generate Neural Profile Embeddings

In [17]:
# =========================
# INITIALIZE ENCODER
# =========================

profile_encoder = ProfileEncoder(

    input_dim=profile_matrix.shape[1]
)

# =========================
# GENERATE EMBEDDINGS
# =========================

profile_tensor = torch.tensor(

    profile_matrix,

    dtype=torch.float32
)

with torch.no_grad():

    profile_embeddings = profile_encoder(

        profile_tensor
    )

profile_embeddings = (

    profile_embeddings
    .numpy()
)

print(
    profile_embeddings.shape
)

(59946, 128)


# Step 16 — Multimodal User Embedding Fusion

Combine semantic and structured embeddings into one unified representation.

In [18]:
# =========================
# CONCATENATE EMBEDDINGS
# =========================

final_user_embeddings = np.concatenate(

    [

        essay_embeddings,

        profile_embeddings
    ],

    axis=1
)

print(
    final_user_embeddings.shape
)

(59946, 512)


# Step 17 — Normalize Final Embeddings

Normalize vectors for cosine similarity retrieval.

In [19]:
# =========================
# NORMALIZATION
# =========================

faiss.normalize_L2(

    final_user_embeddings
)

print(
    "Embeddings normalized."
)

Embeddings normalized.


# Step 18 — Save Final User Embeddings

In [20]:
# =========================
# SAVE FINAL EMBEDDINGS
# =========================

np.save(

    "final_user_embeddings.npy",

    final_user_embeddings
)

print(
    "Final embeddings saved."
)

Final embeddings saved.


# Step 19 — Import FAISS

Load industrial vector similarity search engine.

In [21]:
# =========================
# IMPORT FAISS
# =========================

import faiss

print(
    faiss.__version__
)

1.13.2


# Step 20 — Determine Embedding Dimension

In [22]:
# =========================
# EMBEDDING DIMENSION
# =========================

embedding_dimension = (

    final_user_embeddings.shape[1]
)

print(
    embedding_dimension
)

512


# Step 21 — Create FAISS Index

Initialize vector similarity retrieval engine.

In [23]:
# =========================
# CREATE INDEX
# =========================

index = faiss.IndexFlatIP(

    embedding_dimension
)

print(
    "FAISS index created."
)

FAISS index created.


# Step 22 — Add User Embeddings to FAISS

In [24]:
# =========================
# ADD VECTORS
# =========================

index.add(

    final_user_embeddings.astype(
        "float32"
    )
)

print(

    "Total vectors:",

    index.ntotal
)

Total vectors: 59946


# Step 23 — Test Semantic Similarity Search

In [25]:
# =========================
# TEST SEARCH
# =========================

test_user_id = 0

query_vector = np.array(

    [

        final_user_embeddings[
            test_user_id
        ]

    ],

    dtype="float32"
)

distances, indices = index.search(

    query_vector,

    10
)

print(
    "Candidate IDs:"
)

print(indices)

print(
    "\nSimilarity Scores:"
)

print(distances)

Candidate IDs:
[[    0 49987 19141   235 58363 16742 55246 35254 11273 28446]]

Similarity Scores:
[[1.         0.7699327  0.7698292  0.7688361  0.7656162  0.76404107
  0.7624048  0.75997436 0.7590728  0.7582644 ]]


# Step 24 — Remove Self Matches

In [26]:
# =========================
# FILTER SELF
# =========================

recommended_users = [

    idx

    for idx in indices[0]

    if idx != test_user_id
]

print(
    recommended_users
)

[49987, 19141, 235, 58363, 16742, 55246, 35254, 11273, 28446]


# Step 25 — Candidate Retrieval Function

In [27]:
# =========================
# RETRIEVAL FUNCTION
# =========================

def retrieve_candidates(

    user_id,

    top_k=50
):

    query_vector = np.array(

        [

            final_user_embeddings[
                user_id
            ]

        ],

        dtype="float32"
    )

    distances, indices = index.search(

        query_vector,

        top_k + 1
    )

    candidates = [

        idx

        for idx in indices[0]

        if idx != user_id
    ]

    return candidates[:top_k]

In [28]:
retrieve_candidates(0)

[49987,
 19141,
 235,
 58363,
 16742,
 55246,
 35254,
 11273,
 28446,
 18358,
 57138,
 16251,
 55463,
 27712,
 46085,
 16307,
 13364,
 12565,
 51100,
 56034,
 42863,
 13394,
 50422,
 24771,
 3648,
 25586,
 22173,
 14700,
 7113,
 41183,
 45999,
 506,
 11575,
 15968,
 19926,
 5817,
 32374,
 25333,
 45267,
 26969,
 11383,
 19813,
 28577,
 7125,
 20800,
 23881,
 24151,
 1106,
 19890,
 28410]

# Step 26 — Save FAISS Retrieval Index

In [29]:
# =========================
# SAVE INDEX
# =========================

faiss.write_index(

    index,

    "dating_faiss_index.bin"
)

print(
    "FAISS index saved."
)

FAISS index saved.


STEP 27 — INSPECT MATCHES

In [30]:
def inspect_match(

    user_a,

    user_b
):

    print("=" * 50)

    print(f"USER {user_a}")

    print("=" * 50)

    print(

        df.iloc[user_a][
            "combined_essays"
        ][:1000]
    )

    print("\n")

    print("=" * 50)

    print(f"USER {user_b}")

    print("=" * 50)

    print(

        df.iloc[user_b][
            "combined_essays"
        ][:1000]
    )

In [31]:
inspect_match(0, 166)

USER 0
about me i would love to think that i was some some kind of intellectual either the dumbest smart guy or the smartest dumb guy can t say i can tell the difference i love to talk about ideas and concepts i forge odd metaphors instead of reciting cliches like the simularities between a friend of mine s house and an underwater salt mine my favorite word is salt by the way weird choice i know to me most things in life are better as metaphors i seek to make myself a little better everyday in some productively lazy way got tired of tying my shoes considered hiring a five year old but would probably have to tie both of our shoes decided to only wear leather shoes dress shoes about you you love to have really serious really deep conversations about really silly stuff you have to be willing to snap me out of a light hearted rant with a kiss you don t have to be funny but you have to be able to make me laugh you should be able to bend spoons with your mind and telepathically make me smile

# Step 28 — Extract Important Semantic Keywords

Identify important personality/interests from essays./// only for me 

In [32]:
# ========================= ////only for me
# TF-IDF KEYWORDS
# =========================

from sklearn.feature_extraction.text import TfidfVectorizer

In [33]:
# ========================= ///only for me 
# BUILD TF-IDF VECTORIZER
# =========================

tfidf_vectorizer = TfidfVectorizer(

    stop_words="english",

    max_features=5000
)

tfidf_matrix = tfidf_vectorizer.fit_transform(

    df["combined_essays"]
)

# Step 29 — Extract User Keywords  //only for mr

In [34]:
# only for me
def extract_keywords(

    user_id,

    top_n=15
):

    user_vector = tfidf_matrix[
        user_id
    ]

    feature_names = np.array(

        tfidf_vectorizer.get_feature_names_out()
    )

    scores = user_vector.toarray().flatten()

    top_indices = scores.argsort()[-top_n:][::-1]

    keywords = feature_names[
        top_indices
    ]

    return keywords

In [35]:
extract_keywords(0)

array(['shoes', 'simplicity', 'salt', 'tired', 'lazy', 'game', 'better',
       'video', 'want', 'half', 'able', 'way', 'mice', 'blend',
       'domestic'], dtype=object)

# Step 30 — Compare Match Keywords //only for me 

In [36]:
# only for me
def compare_keywords(

    user_a,

    user_b
):

    keywords_a = set(

        extract_keywords(user_a)
    )

    keywords_b = set(

        extract_keywords(user_b)
    )

    common = (

        keywords_a.intersection(
            keywords_b
        )
    )

    print(
        f"USER {user_a} KEYWORDS:\n"
    )

    print(
        keywords_a
    )

    print("\n")

    print(
        f"USER {user_b} KEYWORDS:\n"
    )

    print(
        keywords_b
    )

    print("\n")

    print(
        "COMMON SIGNALS:\n"
    )

    print(common)

In [37]:
# only for me
compare_keywords(

    0,

    166
)

USER 0 KEYWORDS:

{'salt', 'able', 'want', 'mice', 'video', 'domestic', 'half', 'simplicity', 'better', 'way', 'tired', 'lazy', 'game', 'shoes', 'blend'}


USER 166 KEYWORDS:

{'awful', 'bored', 'pretty', 'say', 'saturday', 'dunno', 'lazy', 'sake', 'flexible', 'cold', 'surfing', 'pursue', 'nearly', 'fun', 'exciting'}


COMMON SIGNALS:

{'lazy'}


USER 0:

deep conversations about silly stuff

USER 166:

making nearly any situation fun

VERY similar personality energy

SentenceTransformer understands this.
FOR EXAMPLE 🔥

Instead of:

COMMON SIGNALS:
{'lazy'}

System should say:


Both users show:
- playful conversational energy
- humorous communication style
- introspective thinking
- relaxed personality tone

THAT is:

actual semantic explainability



# Step 31 — Reciprocal Compatibility Function

Measure mutual compatibility between two users.

In [38]:
# =========================
# RECIPROCAL SIMILARITY
# =========================

from sklearn.metrics.pairwise import cosine_similarity

In [39]:
def reciprocal_similarity(

    user_a,

    user_b
):

    vector_a = final_user_embeddings[
        user_a
    ].reshape(1, -1)

    vector_b = final_user_embeddings[
        user_b
    ].reshape(1, -1)

    similarity_ab = cosine_similarity(

        vector_a,

        vector_b
    )[0][0]

    similarity_ba = cosine_similarity(

        vector_b,

        vector_a
    )[0][0]

    reciprocal_score = (

        similarity_ab

        +

        similarity_ba

    ) / 2

    return reciprocal_score

# Step 32 — Test Reciprocal Compatibility

In [40]:
reciprocal_similarity(

    0,

    166
)

0.7256943583488464

# Step 33 — Reciprocal Candidate Ranking

In [41]:
def reciprocal_rank(

    user_id,

    candidate_ids
):

    scored_candidates = []

    for candidate in candidate_ids:

        score = reciprocal_similarity(

            user_id,

            candidate
        )

        scored_candidates.append(

            (
                candidate,
                score
            )
        )

    ranked = sorted(

        scored_candidates,

        key=lambda x: x[1],

        reverse=True
    )

    return ranked

# Step 34 — Test Reciprocal Ranking

In [42]:
candidates = retrieve_candidates(

    0,

    top_k=20
)

ranked_matches = reciprocal_rank(

    0,

    candidates
)

ranked_matches[:10]

[(49987, 0.769932746887207),
 (19141, 0.769829273223877),
 (235, 0.768836259841919),
 (58363, 0.7656161785125732),
 (16742, 0.7640411257743835),
 (55246, 0.7624049186706543),
 (35254, 0.759974479675293),
 (11273, 0.7590728998184204),
 (28446, 0.7582645416259766),
 (18358, 0.7580136060714722)]

# Step 35 — Semantic Match Explanation

In [43]:
def explain_match(

    user_a,

    user_b
):

    print("=" * 60)

    print(
        f"MATCH EXPLANATION: {user_a} ↔ {user_b}"
    )

    print("=" * 60)

    score = reciprocal_similarity(

        user_a,

        user_b
    )

    print(
        f"\nReciprocal Score: {score:.4f}"
    )

    print("\n")

    compare_keywords(

        user_a,

        user_b
    )

In [44]:
explain_match(

    0,

    166
)

MATCH EXPLANATION: 0 ↔ 166

Reciprocal Score: 0.7257


USER 0 KEYWORDS:

{'salt', 'able', 'want', 'mice', 'video', 'domestic', 'half', 'simplicity', 'better', 'way', 'tired', 'lazy', 'game', 'shoes', 'blend'}


USER 166 KEYWORDS:

{'awful', 'bored', 'pretty', 'say', 'saturday', 'dunno', 'lazy', 'sake', 'flexible', 'cold', 'surfing', 'pursue', 'nearly', 'fun', 'exciting'}


COMMON SIGNALS:

{'lazy'}


# Step 36 — High-Confidence Match Filtering

In [45]:
def high_confidence_matches(

    ranked_matches,

    threshold=0.72
):

    filtered = [

        match

        for match in ranked_matches

        if match[1] >= threshold
    ]

    return filtered

In [46]:
elite_matches = high_confidence_matches(

    ranked_matches
)

elite_matches[:5]

[(49987, 0.769932746887207),
 (19141, 0.769829273223877),
 (235, 0.768836259841919),
 (58363, 0.7656161785125732),
 (16742, 0.7640411257743835)]

# Step 37 — Dynamic Preference Memory

Store evolving semantic preferences for every user.

In [47]:
# =========================
# USER PREFERENCE MEMORY
# =========================

user_preference_memory = defaultdict(

    lambda: defaultdict(float)
)

print(
    "Preference memory initialized."
)

Preference memory initialized.


# Step 38 — Semantic Interest Definitions

In [48]:
# =========================
# INTEREST PROTOTYPES
# =========================

interest_prototypes = {

    "humor":

    "funny humor playful jokes sarcasm witty laugh",

    "travel":

    "travel adventure exploring backpacking tourism",

    "fitness":

    "gym workout fitness lifting healthy sports",

    "music":

    "music concerts bands singing instruments",

    "books":

    "books reading literature philosophy novels",

    "art":

    "art painting creativity photography museums",

    "deep_talks":

    "deep conversations psychology emotions introspection",

    "outdoors":

    "hiking camping mountains beaches surfing",

    "technology":

    "technology programming software ai engineering",

    "food":

    "food cooking restaurants coffee cuisine",

    "movies":

    "movies cinema netflix acting films",

    "spirituality":

    "meditation spirituality mindfulness self growth"
}

# Step 39 — Generate Interest Embeddings

In [49]:
# =========================
# INTEREST EMBEDDINGS
# =========================

interest_embeddings = {}

for interest, text in (

    interest_prototypes.items()
):

    embedding = embedding_model.encode(
        text
    )

    interest_embeddings[
        interest
    ] = embedding

print(
    len(interest_embeddings)
)

12


# Step 40 — Semantic Interest Extraction

In [50]:
# =========================
# INTEREST EXTRACTION
# =========================

from sklearn.metrics.pairwise import cosine_similarity

In [51]:
def semantic_interest_extraction(

    text,

    threshold=0.35
):

    text_embedding = embedding_model.encode(
        text
    )

    detected = []

    for interest, embedding in (

        interest_embeddings.items()
    ):

        similarity = cosine_similarity(

            [text_embedding],

            [embedding]

        )[0][0]

        if similarity >= threshold:

            detected.append(

                (
                    interest,
                    similarity
                )
            )

    detected = sorted(

        detected,

        key=lambda x: x[1],

        reverse=True
    )

    return detected

In [52]:
semantic_interest_extraction(

    df.iloc[0]["combined_essays"]
)

[]

# Step 41 — Learn From Right Swipes

In [53]:
# =========================
# RIGHT SWIPE LEARNING
# =========================

user_right_swipes = defaultdict(int)

user_total_swipes = defaultdict(int)

In [54]:
def learn_from_right_swipe(

    swiper_id,

    target_user_id
):

    text = df.iloc[
        target_user_id
    ][
        "combined_essays"
    ]

    interests = semantic_interest_extraction(
        text
    )

    for interest, score in interests:

        user_preference_memory[
            swiper_id
        ][
            interest
        ] += score

    user_right_swipes[
        swiper_id
    ] += 1

    user_total_swipes[
        swiper_id
    ] += 1

# Step 42 — Learn From Left Swipes

In [55]:
# =========================
# LEFT SWIPE LEARNING
# =========================

def learn_from_left_swipe(

    swiper_id,

    target_user_id
):

    text = df.iloc[
        target_user_id
    ][
        "combined_essays"
    ]

    interests = semantic_interest_extraction(
        text
    )

    for interest, score in interests:

        user_preference_memory[
            swiper_id
        ][
            interest
        ] -= score * 0.5

    user_total_swipes[
        swiper_id
    ] += 1

# Step 43 — Time Decay for Preferences

In [56]:
# =========================
# TIME DECAY
# =========================

def apply_time_decay(

    user_id,

    decay_rate=0.97
):

    preferences = (

        user_preference_memory[
            user_id
        ]
    )

    for interest in preferences:

        preferences[
            interest
        ] *= decay_rate

# Step 44 — Ranked Preference Profile

In [57]:
def ranked_preferences(

    user_id
):

    preferences = (

        user_preference_memory[
            user_id
        ]
    )

    ranked = sorted(

        preferences.items(),

        key=lambda x: x[1],

        reverse=True
    )

    return ranked

In [58]:
learn_from_right_swipe(0, 166)

learn_from_right_swipe(0, 9694)

learn_from_left_swipe(0, 200)

ranked_preferences(0)

[]

# Step 45 — Ideal Match Activation Logic

In [59]:
def eligible_for_ideal_match(

    user_id,

    minimum_total_swipes=20,

    minimum_right_swipes=10
):

    total_swipes = (

        user_total_swipes[
            user_id
        ]
    )

    right_swipes = (

        user_right_swipes[
            user_id
        ]
    )

    return (

        total_swipes >= minimum_total_swipes

        or

        right_swipes >= minimum_right_swipes
    )

In [60]:
eligible_for_ideal_match(0)

False

# Step 46 — Build Interaction Training Dataset

Create positive and negative matchmaking samples.

In [61]:
# =========================
# TRAINING INTERACTIONS
# =========================

interaction_data = []

In [62]:
# =========================
# SIMULATE SWIPE DATA
# =========================

np.random.seed(42)

num_samples = 20000

for _ in range(num_samples):

    user_a = np.random.randint(

        0,

        len(df)
    )

    user_b = np.random.randint(

        0,

        len(df)
    )

    similarity = reciprocal_similarity(

        user_a,

        user_b
    )

    # =========================
    # POSITIVE SAMPLE
    # =========================

    if similarity >= 0.72:

        label = 1

    else:

        label = 0

    interaction_data.append(

        [
            user_a,
            user_b,
            label
        ]
    )

interaction_df = pd.DataFrame(

    interaction_data,

    columns=[
        "user_a",
        "user_b",
        "label"
    ]
)

interaction_df.head()

,user_a,user_b,label
0,56422,15795,0
1,860,38158,0
2,54343,44732,0
3,11284,54886,0
4,6265,16850,0


# Step 47 — Build Neural Training Features

In [63]:
# =========================
# BUILD PAIR FEATURES
# =========================

X = []

y = []

for _, row in interaction_df.iterrows():

    user_a = row["user_a"]

    user_b = row["user_b"]

    label = row["label"]

    embedding_a = final_user_embeddings[
        user_a
    ]

    embedding_b = final_user_embeddings[
        user_b
    ]

    combined = np.concatenate(

        [
            embedding_a,
            embedding_b
        ]
    )

    X.append(combined)

    y.append(label)

X = np.array(X)

y = np.array(y)

print(X.shape)

print(y.shape)

(20000, 1024)
(20000,)


# Step 48 — Split Training Dataset

In [64]:
from sklearn.model_selection import train_test_split

In [65]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=42
)

# Step 49 — Neural Compatibility Ranking Model

In [66]:
# =========================
# NEURAL MATCH RANKER
# =========================

class MatchRanker(

    nn.Module
):

    def __init__(

        self,

        input_dim
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                512
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                512,
                256
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                256,
                1
            ),

            nn.Sigmoid()
        )

    def forward(

        self,

        x
    ):

        return self.network(x)

# Step 50 — Initialize Neural Ranking System

In [67]:
# =========================
# INITIALIZE MODEL
# =========================

model = MatchRanker(

    input_dim=X.shape[1]
)

criterion = nn.BCELoss()

optimizer = torch.optim.Adam(

    model.parameters(),

    lr=0.001
)

print(
    model
)

MatchRanker(
  (network): Sequential(
    (0): Linear(in_features=1024, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=256, out_features=1, bias=True)
    (7): Sigmoid()
  )
)


# Step 51 — Prepare Tensor Datasets

In [68]:
X_train_tensor = torch.tensor(

    X_train,

    dtype=torch.float32
)

y_train_tensor = torch.tensor(

    y_train,

    dtype=torch.float32
).unsqueeze(1)

X_test_tensor = torch.tensor(

    X_test,

    dtype=torch.float32
)

y_test_tensor = torch.tensor(

    y_test,

    dtype=torch.float32
).unsqueeze(1)

# Step 52 — Train Neural Match Ranking Network

In [69]:
# =========================
# TRAINING LOOP
# =========================

epochs = 10

for epoch in range(epochs):

    model.train()

    predictions = model(

        X_train_tensor
    )

    loss = criterion(

        predictions,

        y_train_tensor
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    print(

        f"Epoch {epoch+1}"

        f" | Loss: {loss.item():.4f}"
    )

Epoch 1 | Loss: 0.6872
Epoch 2 | Loss: 0.6462
Epoch 3 | Loss: 0.6054
Epoch 4 | Loss: 0.5541
Epoch 5 | Loss: 0.4899
Epoch 6 | Loss: 0.4157
Epoch 7 | Loss: 0.3387
Epoch 8 | Loss: 0.2698
Epoch 9 | Loss: 0.2173
Epoch 10 | Loss: 0.1857


# Step 53 — Evaluate Neural Compatibility Model

In [70]:
from sklearn.metrics import accuracy_score

In [71]:
model.eval()

with torch.no_grad():

    test_predictions = model(

        X_test_tensor
    )

predicted_labels = (

    test_predictions.numpy() > 0.5
).astype(int)

accuracy = accuracy_score(

    y_test,

    predicted_labels
)

print(
    f"Accuracy: {accuracy:.4f}"
)

Accuracy: 0.9645


# Step 54 — Neural Match Prediction

In [72]:
def neural_match_score(

    user_a,

    user_b
):

    # =========================
    # FORCE INTEGER IDS
    # =========================

    user_a = int(user_a)

    user_b = int(user_b)

    # =========================
    # FETCH EMBEDDINGS
    # =========================

    embedding_a = final_user_embeddings[
        user_a
    ]

    embedding_b = final_user_embeddings[
        user_b
    ]

    # =========================
    # CONCATENATE
    # =========================

    combined = np.concatenate(

        [
            embedding_a,
            embedding_b
        ]
    )

    combined_tensor = torch.tensor(

        combined,

        dtype=torch.float32
    ).unsqueeze(0)

    model.eval()

    with torch.no_grad():

        score = model(

            combined_tensor
        ).item()

    return score

In [73]:
neural_match_score(

    0,

    166
)

0.036600641906261444

# Step 55 — Simulated Behavioral Swipe Engine

Generate realistic swipe interactions using preference memory and compatibility.

In [74]:
def simulate_swipe(

    user_a,

    user_b
):

    compatibility = reciprocal_similarity(

        user_a,

        user_b
    )

    interests_a = semantic_interest_extraction(

        df.iloc[user_a][
            "combined_essays"
        ]
    )

    interests_b = semantic_interest_extraction(

        df.iloc[user_b][
            "combined_essays"
        ]
    )

    interests_a = set(

        [x[0] for x in interests_a]
    )

    interests_b = set(

        [x[0] for x in interests_b]
    )

    overlap = len(

        interests_a.intersection(
            interests_b
        )
    )

    # =========================
    # HUMAN NOISE
    # =========================

    randomness = np.random.normal(

        0,

        0.25
    )

    # =========================
    # IMPERFECT HUMAN BEHAVIOR
    # =========================

    score = (

        compatibility * 0.4

        +

        overlap * 0.15

        +

        randomness
    )

    # =========================
    # PROBABILISTIC SWIPE
    # =========================

    probability = 1 / (

        1 + np.exp(-score)
    )

    swipe = np.random.binomial(

        1,

        probability
    )

    return swipe

In [75]:
# =========================
# PRECOMPUTE USER INTERESTS
# =========================

precomputed_interests = {}

for user_id in range(len(df)):

    interests = semantic_interest_extraction(

        df.iloc[user_id]["combined_essays"]
    )

    precomputed_interests[user_id] = set(

        [x[0] for x in interests]
    )

print("Precomputed interests done ✅")

Precomputed interests done ✅


In [76]:
def simulate_swipe(

    user_a,

    user_b
):

    compatibility = reciprocal_similarity(

        user_a,

        user_b
    )

    interests_a = precomputed_interests[
        user_a
    ]

    interests_b = precomputed_interests[
        user_b
    ]

    overlap = len(

        interests_a.intersection(
            interests_b
        )
    )

    randomness = np.random.normal(

        0,

        0.12
    )

    score = (

        compatibility * 0.4

        +

        overlap * 0.15

        +

        randomness
    )

    probability = 1 / (

        1 + np.exp(-score)
    )

    swipe = np.random.binomial(

        1,

        probability
    )

    return swipe

# Step 56 — Generate Behavioral Interaction Dataset

In [77]:
# =========================
# BEHAVIORAL DATASET
# =========================

behavioral_data = []

num_samples = 30000

for _ in range(num_samples):

    user_a = np.random.randint(

        0,

        len(df)
    )

    user_b = np.random.randint(

        0,

        len(df)
    )

    label = simulate_swipe(

        user_a,

        user_b
    )

    behavioral_data.append(

        [
            user_a,
            user_b,
            label
        ]
    )

behavior_df = pd.DataFrame(

    behavioral_data,

    columns=[
        "user_a",
        "user_b",
        "label"
    ]
)

behavior_df.head()

,user_a,user_b,label
0,7348,575,1
1,25391,22407,0
2,43693,40747,1
3,55532,20244,0
4,1067,17265,1


# Step 57 — Verify Label Distribution

In [78]:
behavior_df["label"].value_counts()

label
1    16550
0    13450
Name: count, dtype: int64

# Step 58 — Build Behavioral Training Features

In [79]:
X = []

y = []

for _, row in behavior_df.iterrows():

    user_a = row["user_a"]

    user_b = row["user_b"]

    label = row["label"]

    embedding_a = final_user_embeddings[
        user_a
    ]

    embedding_b = final_user_embeddings[
        user_b
    ]

    combined = np.concatenate(

        [
            embedding_a,
            embedding_b
        ]
    )

    X.append(combined)

    y.append(label)

X = np.array(X)

y = np.array(y)

print(X.shape)

print(y.shape)

(30000, 1024)
(30000,)


# Step 59 — Split Behavioral Dataset

In [80]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=42
)

# Step 60 — Prepare Behavioral Tensors

In [81]:
X_train_tensor = torch.tensor(

    X_train,

    dtype=torch.float32
)

y_train_tensor = torch.tensor(

    y_train,

    dtype=torch.float32
).unsqueeze(1)

X_test_tensor = torch.tensor(

    X_test,

    dtype=torch.float32
)

y_test_tensor = torch.tensor(

    y_test,

    dtype=torch.float32
).unsqueeze(1)

# Step 61 — Reinitialize Neural Ranker

In [82]:
model = MatchRanker(

    input_dim=X.shape[1]
)

criterion = nn.BCELoss()

optimizer = torch.optim.Adam(

    model.parameters(),

    lr=0.001
)

# Step 62 — Train Behavioral Neural Ranker

In [83]:
epochs = 10

for epoch in range(epochs):

    model.train()

    predictions = model(

        X_train_tensor
    )

    loss = criterion(

        predictions,

        y_train_tensor
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    print(

        f"Epoch {epoch+1}"

        f" | Loss: {loss.item():.4f}"
    )

Epoch 1 | Loss: 0.6930
Epoch 2 | Loss: 0.6898
Epoch 3 | Loss: 0.6881
Epoch 4 | Loss: 0.6886
Epoch 5 | Loss: 0.6887
Epoch 6 | Loss: 0.6882
Epoch 7 | Loss: 0.6877
Epoch 8 | Loss: 0.6878
Epoch 9 | Loss: 0.6880
Epoch 10 | Loss: 0.6880


# Step 63 — Evaluate Behavioral Ranker

In [84]:
model.eval()

with torch.no_grad():

    predictions = model(

        X_test_tensor
    )

predicted_labels = (

    predictions.numpy() > 0.5
).astype(int)

accuracy = accuracy_score(

    y_test,

    predicted_labels
)

print(
    f"Accuracy: {accuracy:.4f}"
)

Accuracy: 0.5568


# Step 64 — Test Behavioral Match Predictions

In [85]:
neural_match_score(

    0,

    166
)

0.5432500839233398

# Step 65 — Build Dynamic User Preference Vector

Convert behavioral preference memory into vector representation.

In [86]:
# =========================
# USER PREFERENCE VECTOR
# =========================

def build_preference_vector(

    user_id
):

    preferences = (

        user_preference_memory[
            user_id
        ]
    )

    vector = np.zeros(

        len(
            interest_prototypes
        )
    )

    interest_names = list(

        interest_prototypes.keys()
    )

    for i, interest in enumerate(

        interest_names
    ):

        vector[i] = preferences.get(
            interest,
            0
        )

    return vector

# Step 66 — Behavioral Preference Alignment

In [87]:
def behavioral_alignment(

    user_id,

    candidate_id
):

    user_vector = build_preference_vector(
        user_id
    )

    candidate_interests = semantic_interest_extraction(

        df.iloc[candidate_id][
            "combined_essays"
        ]
    )

    candidate_vector = np.zeros(

        len(
            interest_prototypes
        )
    )

    interest_names = list(

        interest_prototypes.keys()
    )

    for interest, score in (

        candidate_interests
    ):

        if interest in interest_names:

            idx = interest_names.index(
                interest
            )

            candidate_vector[idx] = score

    similarity = cosine_similarity(

        [user_vector],

        [candidate_vector]

    )[0][0]

    return similarity

# Step 67 — Final Matchmaking Score

In [88]:
def final_match_score(

    user_id,

    candidate_id
):

    # =========================
    # NEURAL COMPATIBILITY
    # =========================

    neural_score = neural_match_score(

        user_id,

        candidate_id
    )

    # =========================
    # RECIPROCAL COMPATIBILITY
    # =========================

    reciprocal_score = reciprocal_similarity(

        user_id,

        candidate_id
    )

    # =========================
    # BEHAVIORAL ALIGNMENT
    # =========================

    behavior_score = behavioral_alignment(

        user_id,

        candidate_id
    )

    # =========================
    # FINAL SCORE
    # =========================

    final_score = (

        neural_score * 0.5

        +

        reciprocal_score * 0.3

        +

        behavior_score * 0.2
    )

    return final_score

# Step 68 — Retrieve High-Confidence Candidates

In [89]:
def retrieve_safe_matches(

    user_id,

    top_k=100
):

    candidates = retrieve_candidates(

        user_id,

        top_k=top_k
    )

    scored = []

    for candidate in candidates:

        score = final_match_score(

            user_id,

            candidate
        )

        scored.append(

            (
                candidate,
                score
            )
        )

    scored = sorted(

        scored,

        key=lambda x: x[1],

        reverse=True
    )

    return scored

# Step 69 — Exploration Match Strategy

In [90]:
def exploration_candidates(

    user_id,

    top_k=20
):

    random_users = np.random.choice(

        len(df),

        top_k,

        replace=False
    )

    scored = []

    for candidate in random_users:

        if candidate == user_id:

            continue

        score = final_match_score(

            user_id,

            candidate
        )

        scored.append(

            (
                candidate,
                score
            )
        )

    scored = sorted(

        scored,

        key=lambda x: x[1],

        reverse=True
    )

    return scored

# Step 70 — Daily Perfect Match Generator

In [91]:
def generate_daily_perfect_match(

    user_id
):

    if not eligible_for_ideal_match(
        user_id
    ):

        return "Not enough swipe data yet."

    safe_candidates = retrieve_safe_matches(

        user_id,

        top_k=50
    )

    perfect_match = safe_candidates[0]

    return perfect_match

In [92]:
generate_daily_perfect_match(0)

'Not enough swipe data yet.'

# Step 71 — Weekly Elite Match Generator

In [93]:
def generate_weekly_elite_matches(

    user_id
):

    if not eligible_for_ideal_match(
        user_id
    ):

        return "Not enough swipe data yet."

    # =========================
    # SAFE MATCHES
    # =========================

    safe_matches = retrieve_safe_matches(

        user_id,

        top_k=20
    )

    # =========================
    # EXPLORATION MATCHES
    # =========================

    explore_matches = exploration_candidates(

        user_id,

        top_k=20
    )

    # =========================
    # FINAL WEEKLY MATCHES
    # =========================

    weekly_matches = [

        safe_matches[0],     # safest

        safe_matches[3],     # diverse safe

        explore_matches[0]   # exploratory
    ]

    return weekly_matches

In [94]:
generate_weekly_elite_matches(0)

'Not enough swipe data yet.'

# Step 72 — Explain Perfect Match

In [95]:
def explain_perfect_match(

    user_id,

    candidate_id
):

    user_id = int(user_id)

    candidate_id = int(candidate_id)

    print("=" * 60)

    print(
        f"PERFECT MATCH EXPLANATION"
    )

    print("=" * 60)

    neural_score = neural_match_score(

        user_id,

        candidate_id
    )

    reciprocal_score = reciprocal_similarity(

        user_id,

        candidate_id
    )

    behavior_score = behavioral_alignment(

        user_id,

        candidate_id
    )

    print(
        f"\nNeural Compatibility: {neural_score:.4f}"
    )

    print(
        f"Reciprocal Compatibility: {reciprocal_score:.4f}"
    )

    print(
        f"Behavioral Alignment: {behavior_score:.4f}"
    )

    print("\n")

    compare_keywords(

        user_id,

        candidate_id
    )

In [97]:
match = generate_daily_perfect_match(0)

candidate_id = match[0]

explain_perfect_match(

    0,

    candidate_id
)

ValueError: invalid literal for int() with base 10: 'N'

In [98]:
generate_daily_perfect_match(0)

'Not enough swipe data yet.'

In [99]:
explain_perfect_match(
    0,
    candidate_id
)

ValueError: invalid literal for int() with base 10: 'N'

In [100]:
for user_id in [0, 10, 50, 100]:

    print("\n")

    print("=" * 70)

    print(f"USER {user_id}")

    print("=" * 70)

    match = generate_daily_perfect_match(
        user_id
    )

    print(match)



USER 0
Not enough swipe data yet.


USER 10
Not enough swipe data yet.


USER 50
Not enough swipe data yet.


USER 100
Not enough swipe data yet.


In [101]:
torch.save(

    model.state_dict(),

    "match_ranker.pth"
)

In [102]:
np.save(

    "final_user_embeddings.npy",

    final_user_embeddings
)

In [103]:
faiss.write_index(

    index,

    "dating_faiss.index"
)

In [104]:
# =========================
# DYNAMIC USER EMBEDDINGS
# =========================

dynamic_user_embeddings = {

    user_id: final_user_embeddings[
        user_id
    ].copy()

    for user_id in range(
        len(df)
    )
}

print(
    "Dynamic embedding store initialized ✅"
)

Dynamic embedding store initialized ✅


In [105]:
register_right_swipe(
    0,
    166
)

store_recent_behavior(
    0,
    166,
    "right"
)

register_right_swipe(
    0,
    9694
)

store_recent_behavior(
    0,
    9694,
    "right"
)

NameError: name 'register_right_swipe' is not defined

In [106]:
adaptive_vector = adaptive_user_embedding(
    0
)

adaptive_vector[:10]

NameError: name 'adaptive_user_embedding' is not defined

# Step 74 — Online Embedding Adaptation

In [107]:
def update_user_embedding(

    user_id,

    target_user_id,

    interaction_strength=0.1
):

    # =========================
    # CURRENT USER VECTOR
    # =========================

    user_vector = dynamic_user_embeddings[
        user_id
    ]

    # =========================
    # TARGET VECTOR
    # =========================

    target_vector = final_user_embeddings[
        target_user_id
    ]

    # =========================
    # MOVE USER VECTOR
    # TOWARD TARGET VECTOR
    # =========================

    updated_vector = (

        user_vector

        +

        interaction_strength

        *

        (
            target_vector
            -
            user_vector
        )
    )

    # =========================
    # NORMALIZE VECTOR
    # =========================

    updated_vector = updated_vector / np.linalg.norm(

        updated_vector
    )

    dynamic_user_embeddings[
        user_id
    ] = updated_vector

# Step 75 — Right Swipe Dynamic Learning

In [108]:
def register_right_swipe(

    user_id,

    target_user_id
):

    # =========================
    # UPDATE PREFERENCE MEMORY
    # =========================

    learn_from_right_swipe(

        user_id,

        target_user_id
    )

    # =========================
    # UPDATE DYNAMIC EMBEDDING
    # =========================

    update_user_embedding(

        user_id,

        target_user_id,

        interaction_strength=0.08
    )

# Step 76 — Left Swipe Dynamic Learning

In [109]:
def register_left_swipe(

    user_id,

    target_user_id
):

    # =========================
    # UPDATE PREFERENCE MEMORY
    # =========================

    learn_from_left_swipe(

        user_id,

        target_user_id
    )

    # =========================
    # MOVE AWAY FROM TARGET
    # =========================

    user_vector = dynamic_user_embeddings[
        user_id
    ]

    target_vector = final_user_embeddings[
        target_user_id
    ]

    updated_vector = (

        user_vector

        -

        0.05

        *

        (
            target_vector
            -
            user_vector
        )
    )

    updated_vector = updated_vector / np.linalg.norm(

        updated_vector
    )

    dynamic_user_embeddings[
        user_id
    ] = updated_vector

# Step 77 — Recent Interaction Memory

In [110]:
from collections import deque
recent_behavior_memory = {

    user_id: deque(maxlen=20)

    for user_id in range(
        len(df)
    )
}

# Step 78 — Track Recent Behavior

In [111]:
def store_recent_behavior(

    user_id,

    target_user_id,

    interaction_type
):

    recent_behavior_memory[
        user_id
    ].append(

        (
            target_user_id,
            interaction_type
        )
    )

# Step 79 — Build Short-Term Preference Vector

In [112]:
def short_term_interest_vector(

    user_id
):

    memory = recent_behavior_memory[
        user_id
    ]

    if len(memory) == 0:

        return np.zeros(

            final_user_embeddings.shape[1]
        )

    vectors = []

    for target_user_id, interaction in memory:

        if interaction == "right":

            vectors.append(

                final_user_embeddings[
                    target_user_id
                ]
            )

    if len(vectors) == 0:

        return np.zeros(

            final_user_embeddings.shape[1]
        )

    mean_vector = np.mean(

        vectors,

        axis=0
    )

    return mean_vector

# Step 80 — Final Adaptive User Embedding

In [113]:
def adaptive_user_embedding(

    user_id
):

    long_term = dynamic_user_embeddings[
        user_id
    ]

    short_term = short_term_interest_vector(
        user_id
    )

    adaptive_embedding = (

        long_term * 0.7

        +

        short_term * 0.3
    )

    adaptive_embedding = adaptive_embedding / np.linalg.norm(

        adaptive_embedding
    )

    return adaptive_embedding

# Step 81 — Simulate Preference Drift

In [114]:
register_right_swipe(

    0,

    166
)

store_recent_behavior(

    0,

    166,

    "right"
)

register_right_swipe(

    0,

    9694
)

store_recent_behavior(

    0,

    9694,

    "right"
)

adaptive_vector = adaptive_user_embedding(
    0
)

adaptive_vector[:10]

array([-0.03498142, -0.08901902,  0.07522693, -0.00394531,  0.06302317,
       -0.03078438,  0.11081295,  0.04167689,  0.0172914 ,  0.03042602],
      dtype=float32)

# Step 82 — Build Sequential Swipe Histories

In [115]:
# =========================
# USER SWIPE SEQUENCES
# =========================

user_swipe_sequences = {

    user_id: []

    for user_id in range(
        len(df)
    )
}

# Step 83 — Record Sequential Swipes

In [116]:
def add_swipe_sequence(

    user_id,

    target_user_id
):

    user_swipe_sequences[
        user_id
    ].append(

        target_user_id
    )

In [117]:
add_swipe_sequence(0, 166)

add_swipe_sequence(0, 9694)

add_swipe_sequence(0, 712)

user_swipe_sequences[0]

[166, 9694, 712]

# Step 84 — Prepare Fixed-Length Sequences

In [118]:
MAX_SEQUENCE_LENGTH = 20

In [119]:
def prepare_sequence(

    user_id
):

    sequence = user_swipe_sequences[
        user_id
    ]

    if len(sequence) >= MAX_SEQUENCE_LENGTH:

        sequence = sequence[
            -MAX_SEQUENCE_LENGTH:
        ]

    else:

        padding = [

            0
        ] * (

            MAX_SEQUENCE_LENGTH
            -
            len(sequence)
        )

        sequence = padding + sequence

    return sequence

# Step 85 — Convert Swipe History into Embedding Sequences

In [120]:
def sequence_embedding(

    user_id
):

    sequence = prepare_sequence(
        user_id
    )

    embeddings = []

    for target_user_id in sequence:

        embedding = final_user_embeddings[
            target_user_id
        ]

        embeddings.append(
            embedding
        )

    embeddings = np.array(
        embeddings
    )

    return embeddings

In [121]:
sequence_embedding(0).shape

(20, 512)

# Step 86 — Sequential Transformer Recommender

In [122]:
class SwipeTransformer(

    nn.Module
):

    def __init__(

        self,

        embedding_dim,

        num_heads=4,

        hidden_dim=256,

        num_layers=2
    ):

        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(

            d_model=embedding_dim,

            nhead=num_heads,

            dim_feedforward=hidden_dim,

            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(

            encoder_layer,

            num_layers=num_layers
        )

        self.output_layer = nn.Linear(

            embedding_dim,

            embedding_dim
        )

    def forward(

        self,

        x
    ):

        transformed = self.transformer(
            x
        )

        final_state = transformed[
            :,
            -1,
            :
        ]

        output = self.output_layer(
            final_state
        )

        return output

# Step 87 — Initialize Sequential Transformer

In [123]:
embedding_dim = final_user_embeddings.shape[1]

In [124]:
sequence_model = SwipeTransformer(

    embedding_dim=embedding_dim
)

# Step 88 — Test Sequential Intelligence

In [125]:
sequence = sequence_embedding(0)

In [126]:
sequence_tensor = torch.tensor(

    sequence,

    dtype=torch.float32
).unsqueeze(0)

In [127]:
predicted_preference = sequence_model(

    sequence_tensor
)

In [128]:
predicted_preference.shape

torch.Size([1, 512])

# Step 89 — Predict Future-Compatible Matches

In [129]:
def transformer_recommendations(

    user_id,

    top_k=10
):

    sequence = sequence_embedding(
        user_id
    )

    sequence_tensor = torch.tensor(

        sequence,

        dtype=torch.float32
    ).unsqueeze(0)

    with torch.no_grad():

        predicted_embedding = sequence_model(

            sequence_tensor
        ).numpy()

    similarities = cosine_similarity(

        predicted_embedding,

        final_user_embeddings
    )[0]

    top_indices = np.argsort(

        similarities
    )[::-1][:top_k]

    return top_indices

In [130]:
transformer_recommendations(0)

array([51922, 50426, 41646,  7530, 42462, 36680, 12995, 59541, 11411,
       37570])

# Step 90 — Build Transformer Training Sequences

In [131]:
sequence_X = []
sequence_y = []

for user_id in range(len(df)):

    sequence = user_swipe_sequences[user_id]

    if len(sequence) < 2:
        continue

    for i in range(1, len(sequence)):

        history = sequence[:i]

        target = sequence[i]

        if len(history) >= MAX_SEQUENCE_LENGTH:
            history = history[-MAX_SEQUENCE_LENGTH:]

        else:
            padding = [0] * (
                MAX_SEQUENCE_LENGTH - len(history)
            )
            history = padding + history

        history_embeddings = []

        for item in history:

            history_embeddings.append(
                final_user_embeddings[item]
            )

        sequence_X.append(
            np.array(history_embeddings)
        )

        sequence_y.append(
            final_user_embeddings[target]
        )

sequence_X = np.array(sequence_X)
sequence_y = np.array(sequence_y)

print(sequence_X.shape)
print(sequence_y.shape)

(2, 20, 512)
(2, 512)


In [132]:
X_seq_tensor = torch.tensor(

    sequence_X,

    dtype=torch.float32
)

y_seq_tensor = torch.tensor(

    sequence_y,

    dtype=torch.float32
)

In [133]:
sequence_model = SwipeTransformer(

    embedding_dim=embedding_dim
)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(

    sequence_model.parameters(),

    lr=0.001
)

# Step 93 — Train Sequential Transformer

In [134]:
for epoch in range(10):

    optimizer.zero_grad()

    predictions = sequence_model(
        X_seq_tensor
    )

    loss = criterion(

        predictions,

        y_seq_tensor
    )

    loss.backward()

    optimizer.step()

    print(

        f"Epoch {epoch+1} | Loss: {loss.item():.4f}"
    )

Epoch 1 | Loss: 0.3301
Epoch 2 | Loss: 0.2452
Epoch 3 | Loss: 0.1008
Epoch 4 | Loss: 0.0802
Epoch 5 | Loss: 0.0782
Epoch 6 | Loss: 0.0692
Epoch 7 | Loss: 0.0558
Epoch 8 | Loss: 0.0518
Epoch 9 | Loss: 0.0489
Epoch 10 | Loss: 0.0513


In [135]:
transformer_recommendations(0)

array([41287,  5567, 49692,  5791, 54793, 22257, 16536,  7347, 26641,
       29553])

# Step 95 — Install CLIP Dependencies

In [136]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-002u0cup
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-002u0cup

  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369549 sha256=c893b5be6f145261374b44083ef3fb2a2f87def90b933d211a3e76d398f450fb
  Stored in directory: /tmp/pip-ephem-wheel-cache-88lw1j32/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [clip]


# Step 96 — Load CLIP Model

In [137]:
import clip
from PIL import Image

In [138]:
!pip install transformers pillow

In [139]:
from transformers import CLIPProcessor, CLIPModel

from PIL import Image

import requests

In [140]:
clip_model = CLIPModel.from_pretrained(

    "openai/clip-vit-base-patch32"
)

clip_processor = CLIPProcessor.from_pretrained(

    "openai/clip-vit-base-patch32"
)

print(
    "CLIP loaded ✅"
)

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP loaded ✅


In [141]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [142]:
clip_model, preprocess = clip.load(

    "ViT-B/32",

    device=device
)

100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 115MiB/s]


# Step 97 — Generate Image Embeddings

In [143]:
def generate_image_embedding(

    image_path
):

    image = preprocess(

        Image.open(image_path)

    ).unsqueeze(0).to(device)

    with torch.no_grad():

        embedding = clip_model.encode_image(
            image
        )

    embedding = embedding.cpu().numpy()[0]

    embedding = embedding / np.linalg.norm(
        embedding
    )

    return embedding

# Step 98 — User Image Embedding Store

In [144]:
user_image_embeddings = {}

In [145]:
user_image_embeddings[0] = generate_image_embedding(

    "user0.jpg"
)

FileNotFoundError: [Errno 2] No such file or directory: 'user0.jpg'

# Step 99 — Multimodal User Representation

In [146]:
def multimodal_user_embedding(

    user_id
):

    text_embedding = adaptive_user_embedding(
        user_id
    )

    image_embedding = user_image_embeddings.get(

        user_id,

        np.zeros_like(text_embedding)
    )

    combined = np.concatenate(

        [
            text_embedding,
            image_embedding
        ]
    )

    combined = combined / np.linalg.norm(
        combined
    )

    return combined

# Step 100 — Multimodal Compatibility Score

In [147]:
def multimodal_match_score(

    user_a,

    user_b
):

    embedding_a = multimodal_user_embedding(
        user_a
    )

    embedding_b = multimodal_user_embedding(
        user_b
    )

    similarity = cosine_similarity(

        [embedding_a],

        [embedding_b]

    )[0][0]

    return similarity

In [148]:
multimodal_match_score(

    0,

    166
)

0.8489232347116756

# Step 101 — Visual Preference Adaptation

In [149]:
visual_preference_memory = {

    user_id: np.zeros(

        512
    )

    for user_id in range(
        len(df)
    )
}

In [150]:
def update_visual_preferences(

    user_id,

    target_user_id,

    strength=0.05
):

    if target_user_id not in user_image_embeddings:

        return

    visual_vector = user_image_embeddings[
        target_user_id
    ]

    visual_preference_memory[
        user_id
    ] += (

        visual_vector * strength
    )

# Step 102 — Industrial Multimodal User Embedding

In [151]:
def industrial_user_embedding(

    user_id
):

    adaptive_embedding = adaptive_user_embedding(
        user_id
    )

    visual_embedding = visual_preference_memory[
        user_id
    ]

    final_embedding = (

        adaptive_embedding * 0.8

        +

        visual_embedding * 0.2
    )

    final_embedding = final_embedding / np.linalg.norm(

        final_embedding
    )

    return final_embedding

# Step 103 — User Tower Network

In [152]:
class UserTower(

    nn.Module
):

    def __init__(

        self,

        input_dim,

        embedding_dim=256
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                512
            ),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(
                512,
                embedding_dim
            )
        )

    def forward(

        self,

        x
    ):

        embedding = self.network(x)

        embedding = F.normalize(

            embedding,

            p=2,

            dim=1
        )

        return embedding

# Step 104 — Candidate Tower Network

In [153]:
class CandidateTower(

    nn.Module
):

    def __init__(

        self,

        input_dim,

        embedding_dim=256
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                512
            ),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(
                512,
                embedding_dim
            )
        )

    def forward(

        self,

        x
    ):

        embedding = self.network(x)

        embedding = F.normalize(

            embedding,

            p=2,

            dim=1
        )

        return embedding

# Step 105 — Initialize Two-Tower Retrieval

In [154]:
input_dim = industrial_user_embedding(
    0
).shape[0]

In [155]:
embedding_dim = 256

In [156]:
user_tower = UserTower(

    input_dim,

    embedding_dim
)

candidate_tower = CandidateTower(

    input_dim,

    embedding_dim
)

# Step 106 — Build Retrieval Training Pairs

In [157]:
retrieval_X_user = []

retrieval_X_candidate = []

retrieval_y = []

In [158]:
for _ in range(15000):

    user_a = np.random.randint(

        0,

        len(df)
    )

    user_b = np.random.randint(

        0,

        len(df)
    )

    label = simulate_swipe(

        user_a,

        user_b
    )

    user_embedding = industrial_user_embedding(
        user_a
    )

    candidate_embedding = industrial_user_embedding(
        user_b
    )

    retrieval_X_user.append(
        user_embedding
    )

    retrieval_X_candidate.append(
        candidate_embedding
    )

    retrieval_y.append(
        label
    )

# Step 107 — Prepare Retrieval Tensors

In [159]:
X_user_tensor = torch.tensor(

    np.array(retrieval_X_user),

    dtype=torch.float32
)

X_candidate_tensor = torch.tensor(

    np.array(retrieval_X_candidate),

    dtype=torch.float32
)

y_tensor = torch.tensor(

    retrieval_y,

    dtype=torch.float32
).unsqueeze(1)

# Step 108 — Train Two-Tower Retrieval

In [160]:
optimizer = torch.optim.Adam(

    list(user_tower.parameters())

    +

    list(candidate_tower.parameters()),

    lr=0.001
)

criterion = nn.BCELoss()

In [161]:
epochs = 10

for epoch in range(epochs):

    optimizer.zero_grad()

    user_embeddings = user_tower(

        X_user_tensor
    )

    candidate_embeddings = candidate_tower(

        X_candidate_tensor
    )

    similarities = torch.sum(

        user_embeddings

        *

        candidate_embeddings,

        dim=1,

        keepdim=True
    )

    predictions = torch.sigmoid(
        similarities
    )

    loss = criterion(

        predictions,

        y_tensor
    )

    loss.backward()

    optimizer.step()

    print(

        f"Epoch {epoch+1}"

        f" | Loss: {loss.item():.4f}"
    )

Epoch 1 | Loss: 0.6964
Epoch 2 | Loss: 0.7040
Epoch 3 | Loss: 0.6956
Epoch 4 | Loss: 0.6878
Epoch 5 | Loss: 0.6895
Epoch 6 | Loss: 0.6917
Epoch 7 | Loss: 0.6904
Epoch 8 | Loss: 0.6883
Epoch 9 | Loss: 0.6871
Epoch 10 | Loss: 0.6870


# Step 109 — Generate Candidate Tower Embeddings

In [162]:
candidate_vectors = []

In [163]:
for user_id in range(len(df)):

    embedding = industrial_user_embedding(
        user_id
    )

    tensor = torch.tensor(

        embedding,

        dtype=torch.float32
    ).unsqueeze(0)

    with torch.no_grad():

        vector = candidate_tower(
            tensor
        ).numpy()[0]

    candidate_vectors.append(
        vector
    )

candidate_vectors = np.array(
    candidate_vectors
)

# Step 110 — Industrial FAISS Retrieval Index

In [164]:
dimension = candidate_vectors.shape[1]

In [165]:
industrial_index = faiss.IndexFlatIP(
    dimension
)

In [166]:
industrial_index.add(

    candidate_vectors.astype(
        np.float32
    )
)

# Step 111 — Two-Tower Candidate Retrieval

In [167]:
def industrial_retrieval(

    user_id,

    top_k=10
):

    user_embedding = industrial_user_embedding(
        user_id
    )

    tensor = torch.tensor(

        user_embedding,

        dtype=torch.float32
    ).unsqueeze(0)

    with torch.no_grad():

        query_vector = user_tower(
            tensor
        ).numpy()

    scores, indices = industrial_index.search(

        query_vector.astype(
            np.float32
        ),

        top_k
    )

    return indices[0], scores[0]

In [168]:
industrial_retrieval(0)

(array([ 9977, 20517, 55476, 17271, 11188, 16800, 18608, 18290, 14502,
        54008]),
 array([0.34323373, 0.34234548, 0.34217924, 0.34181625, 0.34167668,
        0.34146348, 0.34141773, 0.34071028, 0.34063864, 0.34015045],
       dtype=float32))

# Step 112 — Streaming Interaction Store

In [169]:
streaming_events = []

# Step 113 — Log Real-Time User Events

In [170]:
import time

In [171]:
def log_event(

    user_id,

    target_user_id,

    event_type
):

    event = {

        "timestamp": time.time(),

        "user_id": user_id,

        "target_user_id": target_user_id,

        "event_type": event_type
    }

    streaming_events.append(
        event
    )

In [172]:
log_event(

    0,

    166,

    "right_swipe"
)

# Step 114 — Interaction Reward System

In [173]:
event_rewards = {

    "right_swipe": 0.10,

    "match": 0.25,

    "message_sent": 0.30,

    "reply_received": 0.40,

    "long_conversation": 0.60,

    "date_success": 1.00,

    "ghosted": -0.30,

    "unmatch": -0.50
}

# Step 115 — Real-Time Embedding Adaptation

In [174]:
def online_embedding_update(

    user_id,

    target_user_id,

    event_type
):

    reward = event_rewards.get(

        event_type,

        0
    )

    target_embedding = industrial_user_embedding(

        target_user_id
    )

    current_embedding = dynamic_user_embeddings[
        user_id
    ]

    updated_embedding = (

        current_embedding

        +

        reward

        *

        (
            target_embedding
            -
            current_embedding
        )
    )

    updated_embedding = updated_embedding / np.linalg.norm(

        updated_embedding
    )

    dynamic_user_embeddings[
        user_id
    ] = updated_embedding

# Step 116 — Streaming Online Learning Pipeline

In [175]:
def process_event(

    user_id,

    target_user_id,

    event_type
):

    # =========================
    # LOG EVENT
    # =========================

    log_event(

        user_id,

        target_user_id,

        event_type
    )

    # =========================
    # UPDATE PREFERENCE MEMORY
    # =========================

    if event_type in [

        "right_swipe",

        "match",

        "message_sent",

        "reply_received",

        "long_conversation"
    ]:

        register_right_swipe(

            user_id,

            target_user_id
        )

        store_recent_behavior(

            user_id,

            target_user_id,

            "right"
        )

    # =========================
    # NEGATIVE SIGNALS
    # =========================

    if event_type in [

        "ghosted",

        "unmatch"
    ]:

        register_left_swipe(

            user_id,

            target_user_id
        )

    # =========================
    # ONLINE EMBEDDING UPDATE
    # =========================

    online_embedding_update(

        user_id,

        target_user_id,

        event_type
    )

In [176]:
process_event(

    0,

    166,

    "message_sent"
)

# Step 117 — Real-Time Adaptive Retrieval

In [177]:
def live_recommendations(

    user_id,

    top_k=10
):

    user_embedding = industrial_user_embedding(
        user_id
    )

    tensor = torch.tensor(

        user_embedding,

        dtype=torch.float32
    ).unsqueeze(0)

    with torch.no_grad():

        query_vector = user_tower(
            tensor
        ).numpy()

    scores, indices = industrial_index.search(

        query_vector.astype(
            np.float32
        ),

        top_k
    )

    return indices[0]

In [178]:
live_recommendations(0)

array([26491, 17812, 54008, 24331, 16800, 19455, 18883, 11188,  9977,
       20904])

# Step 118 — Time Decay Preference Engine

In [179]:
def apply_time_decay(

    decay_factor=0.995
):

    for user_id in dynamic_user_embeddings:

        dynamic_user_embeddings[
            user_id
        ] *= decay_factor

        norm = np.linalg.norm(

            dynamic_user_embeddings[
                user_id
            ]
        )

        if norm > 0:

            dynamic_user_embeddings[
                user_id
            ] /= norm

# Step 119 — Long-Term Match Utility

In [180]:
def long_term_match_score(

    user_a,

    user_b
):

    retrieval_score = multimodal_match_score(

        user_a,

        user_b
    )

    neural_score = neural_match_score(

        user_a,

        user_b
    )

    transformer_candidates = transformer_recommendations(
        user_a,
        top_k=20
    )

    sequential_bonus = 0

    if user_b in transformer_candidates:

        sequential_bonus = 0.15

    final_score = (

        retrieval_score * 0.4

        +

        neural_score * 0.3

        +

        sequential_bonus
    )

    return final_score

# Step 120 — Live Adaptive Match Generator

In [181]:
def generate_live_perfect_match(

    user_id
):

    candidates = live_recommendations(

        user_id,

        top_k=50
    )

    scored = []

    for candidate in candidates:

        if candidate == user_id:

            continue

        score = long_term_match_score(

            user_id,

            candidate
        )

        scored.append(

            (
                candidate,
                score
            )
        )

    scored = sorted(

        scored,

        key=lambda x: x[1],

        reverse=True
    )

    return scored[0]

In [182]:
generate_live_perfect_match(0)

(22301, 0.46585236873968566)

# Step 121 — Exposure Memory System

In [183]:
shown_candidates = {

    user_id: set()

    for user_id in range(
        len(df)
    )
}

# Step 122 — Epsilon-Greedy Exploration Policy

In [184]:
EPSILON = 0.20

In [185]:
def should_explore():

    return np.random.rand() < EPSILON

# Step 123 — High-Confidence Candidate Retrieval

In [186]:
def safe_candidates(

    user_id,

    top_k=50
):

    candidates = live_recommendations(

        user_id,

        top_k=top_k
    )

    filtered = []

    for candidate in candidates:

        if candidate == user_id:

            continue

        if candidate in shown_candidates[
            user_id
        ]:

            continue

        filtered.append(
            candidate
        )

    return filtered

# Step 124 — Exploration Candidate Sampling

In [187]:
def exploration_candidates(

    user_id,

    sample_size=100
):

    random_pool = np.random.choice(

        len(df),

        sample_size,

        replace=False
    )

    scored = []

    for candidate in random_pool:

        if candidate == user_id:

            continue

        if candidate in shown_candidates[
            user_id
        ]:

            continue

        score = long_term_match_score(

            user_id,

            candidate
        )

        novelty_bonus = np.random.uniform(

            0.05,

            0.25
        )

        score += novelty_bonus

        scored.append(

            (
                candidate,
                score
            )
        )

    scored = sorted(

        scored,

        key=lambda x: x[1],

        reverse=True
    )

    return scored

# Step 125 — Contextual Bandit Match Selection

In [188]:
def contextual_bandit_match(

    user_id
):

    # =========================
    # EXPLORATION
    # =========================

    if should_explore():

        candidates = exploration_candidates(
            user_id
        )

        strategy = "exploration"

    # =========================
    # EXPLOITATION
    # =========================

    else:

        candidates = [

            (
                candidate,

                long_term_match_score(
                    user_id,
                    candidate
                )
            )

            for candidate in safe_candidates(
                user_id
            )
        ]

        candidates = sorted(

            candidates,

            key=lambda x: x[1],

            reverse=True
        )

        strategy = "exploitation"

    selected_candidate = candidates[0]

    shown_candidates[
        user_id
    ].add(

        selected_candidate[0]
    )

    return {

        "strategy": strategy,

        "candidate": selected_candidate
    }

In [189]:
contextual_bandit_match(0)

{'strategy': 'exploration', 'candidate': (2461, 0.7093040860339539)}

contextual_bandit_match(0)

In [191]:
def weekly_elite_matches(

    user_id
):

    matches = []

    # =========================
    # SAFE MATCH
    # =========================

    safe_match = contextual_bandit_match(
        user_id
    )

    matches.append(
        safe_match
    )

    # =========================
    # SECOND SAFE
    # =========================

    safe_match_2 = contextual_bandit_match(
        user_id
    )

    matches.append(
        safe_match_2
    )

    # =========================
    # FORCED EXPLORATION
    # =========================

    explore_matches = exploration_candidates(
        user_id
    )

    matches.append(

        {
            "strategy": "forced_exploration",

            "candidate": explore_matches[0]
        }
    )

    return matches

In [192]:
weekly_elite_matches(0)

[{'strategy': 'exploitation', 'candidate': (18586, 0.4726937715246734)},
 {'strategy': 'exploitation', 'candidate': (17271, 0.45399830131776553)},
 {'strategy': 'forced_exploration', 'candidate': (13723, 0.6948206092956499)}]

# Step 127 — Uncertainty-Aware Exploration

In [193]:
candidate_uncertainty = {}

In [194]:
for user_id in range(len(df)):

    candidate_uncertainty[
        user_id
    ] = np.random.uniform(

        0,

        1
    )

In [195]:
def uncertainty_adjusted_score(

    user_id,

    candidate_id
):

    base_score = long_term_match_score(

        user_id,

        candidate_id
    )

    uncertainty = candidate_uncertainty.get(

        candidate_id,

        0
    )

    adjusted_score = (

        base_score

        +

        uncertainty * 0.1
    )

    return adjusted_score

# Step 128 — Industrial Exploration-Aware Matchmaking

In [196]:
def industrial_matchmaking_engine(

    user_id,

    top_k=10
):

    candidates = live_recommendations(

        user_id,

        top_k=100
    )

    scored = []

    for candidate in candidates:

        if candidate == user_id:

            continue

        score = uncertainty_adjusted_score(

            user_id,

            candidate
        )

        scored.append(

            (
                candidate,
                score
            )
        )

    scored = sorted(

        scored,

        key=lambda x: x[1],

        reverse=True
    )

    return scored[:top_k]

In [197]:
industrial_matchmaking_engine(0)

[(18586, 0.5721472631753765),
 (47997, 0.5494431851859594),
 (11602, 0.5424817605681119),
 (11724, 0.541101186260004),
 (6145, 0.5329728685460215),
 (25896, 0.5326203022819037),
 (8861, 0.5305591276699542),
 (54008, 0.5282403241097855),
 (54074, 0.5279815071047),
 (21197, 0.5277780712366413)]

# Step 129 — Install Graph Neural Network Libraries

In [198]:
!pip install torch-geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torch-geometric] [torch-geometric]


# Step 130 — Import Graph Learning Modules

In [199]:
from torch_geometric.data import Data

from torch_geometric.nn import GCNConv

# Step 131 — Build User Interaction Graph

In [200]:
edges = []

In [201]:
for event in streaming_events:

    user_a = event["user_id"]

    user_b = event["target_user_id"]

    if event["event_type"] in [

        "right_swipe",

        "match",

        "message_sent",

        "reply_received",

        "long_conversation"
    ]:

        edges.append(

            [
                user_a,
                user_b
            ]
        )

# Step 132 — Create Graph Edge Index

In [202]:
edge_index = torch.tensor(

    edges,

    dtype=torch.long
).t().contiguous()

In [203]:
edge_index.shape

torch.Size([2, 2])

# Step 133 — Graph Node Features

In [204]:
node_features = []

In [205]:
for user_id in range(len(df)):

    embedding = industrial_user_embedding(
        user_id
    )

    node_features.append(
        embedding
    )

node_features = torch.tensor(

    np.array(node_features),

    dtype=torch.float32
)

# Step 134 — Create Graph Dataset

In [206]:
graph_data = Data(

    x=node_features,

    edge_index=edge_index
)

# Step 135 — Graph Neural Matchmaking Model

In [207]:
class MatchGNN(

    nn.Module
):

    def __init__(

        self,

        input_dim,

        hidden_dim=256
    ):

        super().__init__()

        self.conv1 = GCNConv(

            input_dim,

            hidden_dim
        )

        self.conv2 = GCNConv(

            hidden_dim,

            hidden_dim
        )

    def forward(

        self,

        x,

        edge_index
    ):

        x = self.conv1(

            x,

            edge_index
        )

        x = F.relu(x)

        x = self.conv2(

            x,

            edge_index
        )

        return x

# Step 136 — Initialize Graph Neural Network

In [208]:
input_dim = node_features.shape[1]

In [209]:
gnn_model = MatchGNN(
    input_dim
)

# Step 137 — Train Graph Neural Network

In [210]:
optimizer = torch.optim.Adam(

    gnn_model.parameters(),

    lr=0.001
)

In [211]:
for epoch in range(10):

    optimizer.zero_grad()

    graph_embeddings = gnn_model(

        graph_data.x,

        graph_data.edge_index
    )

    loss = graph_embeddings.norm()

    loss.backward()

    optimizer.step()

    print(

        f"Epoch {epoch+1}"

        f" | Loss: {loss.item():.4f}"
    )

Epoch 1 | Loss: 146.9391
Epoch 2 | Loss: 105.7242
Epoch 3 | Loss: 80.6169
Epoch 4 | Loss: 66.1119
Epoch 5 | Loss: 55.9561
Epoch 6 | Loss: 47.4223
Epoch 7 | Loss: 39.8958
Epoch 8 | Loss: 33.3931
Epoch 9 | Loss: 28.0050
Epoch 10 | Loss: 23.7315


# Step 138 — Graph-Based Compatibility

In [212]:
def graph_match_score(

    user_a,

    user_b
):

    with torch.no_grad():

        graph_embeddings = gnn_model(

            graph_data.x,

            graph_data.edge_index
        )

    embedding_a = graph_embeddings[
        user_a
    ].numpy()

    embedding_b = graph_embeddings[
        user_b
    ].numpy()

    similarity = cosine_similarity(

        [embedding_a],

        [embedding_b]

    )[0][0]

    return similarity

In [213]:
graph_match_score(

    0,

    166
)

0.93999135

# Step 139 — Hybrid Industrial Matchmaking Engine

In [214]:
def hybrid_match_score(

    user_a,

    user_b
):

    multimodal_score = multimodal_match_score(

        user_a,

        user_b
    )

    neural_score = neural_match_score(

        user_a,

        user_b
    )

    graph_score = graph_match_score(

        user_a,

        user_b
    )

    final_score = (

        multimodal_score * 0.35

        +

        neural_score * 0.30

        +

        graph_score * 0.35
    )

    return final_score

# Step 140 — Industrial Hybrid Match Generator

In [215]:
def industrial_hybrid_matches(

    user_id,

    top_k=10
):

    candidates = industrial_retrieval(

        user_id,

        top_k=100
    )[0]

    scored = []

    for candidate in candidates:

        if candidate == user_id:

            continue

        score = hybrid_match_score(

            user_id,

            candidate
        )

        scored.append(

            (
                candidate,
                score
            )
        )

    scored = sorted(

        scored,

        key=lambda x: x[1],

        reverse=True
    )

    return scored[:top_k]

In [216]:
industrial_hybrid_matches(0)

[(27803, 0.7216172711356945),
 (54885, 0.716367115488246),
 (18586, 0.7074907962014426),
 (14502, 0.6963874043784132),
 (8977, 0.6955979760834388),
 (30739, 0.6896732767785466),
 (26260, 0.6845476919296298),
 (48471, 0.6771519220167905),
 (34504, 0.6770025173495604),
 (19221, 0.6763188848128372)]